# Tugas Praktikum - Wisconsin Breast Cancer

Langkah pengerjaan:
1. Memisahkan variabel yang dapat digunakan dan tidak dapat digunakan
2. Encoding kolom `diagnosis`
3. Standarisasi seluruh kolom numerik
4. Stratified split data latih dan data uji dengan rasio 80:20

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
dpath = 'dataset/wbc.csv'
df = pd.read_csv(dpath)
print(f'Ukuran data: {df.shape}')
df.head()

Ukuran data: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
# Cek info dan jumlah data kosong tiap kolom
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

In [4]:
print('Jumlah data kosong tiap kolom:')
print(df.isna().sum(), "\n")
print('Tipe data tiap kolom:')
print(df.dtypes)

Jumlah data kosong tiap kolom:
id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_

## 1. Memisahkan variabel yang dapat digunakan dan tidak dapat digunakan

Variabel yang **tidak dapat digunakan**:
- `id` : hanya nomor identitas pasien, bukan ciri sel tumor
- `Unnamed: 32` : kolom sisa (kosong) dari file csv

Variabel yang **dapat digunakan**:
- `diagnosis` : label/target (M = Malignant, B = Benign)
- 30 kolom fitur numerik hasil pengukuran citra sel (`*_mean`, `*_se`, `*_worst`)

In [5]:
# Daftar variabel yang tidak dapat digunakan
kolom_tidak_dipakai = ['id', 'Unnamed: 32']
print(f'Variabel tidak dapat digunakan: {kolom_tidak_dipakai}\n')

# Buang variabel yang tidak dapat digunakan
df = df.drop(columns=kolom_tidak_dipakai)

# Daftar variabel yang dapat digunakan
label = 'diagnosis'
fitur = [c for c in df.columns if c != label]
print(f'Jumlah variabel dapat digunakan: {df.shape[1]}')
print(f'Label  : {label}')
print(f'Fitur  : {fitur}')
df.head()

Variabel tidak dapat digunakan: ['id', 'Unnamed: 32']

Jumlah variabel dapat digunakan: 31
Label  : diagnosis
Fitur  : ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 2. Encoding pada kolom diagnosis

Nilai teks `M` dan `B` diubah menjadi angka dengan `LabelEncoder` (B = 0, M = 1).

In [6]:
print('=== Sebelum Encoding ===')
print(df[label].value_counts(), "\n")

le = LabelEncoder()
df[label] = le.fit_transform(df[label])

print('=== Setelah Encoding ===')
print(f'Mapping label: {dict(zip(le.classes_.tolist(), le.transform(le.classes_).tolist()))}\n')
print(df[[label]].head(), "\n")
print(df[label].value_counts())

=== Sebelum Encoding ===
diagnosis
B    357
M    212
Name: count, dtype: int64 

=== Setelah Encoding ===
Mapping label: {'B': 0, 'M': 1}

   diagnosis
0          1
1          1
2          1
3          1
4          1 

diagnosis
0    357
1    212
Name: count, dtype: int64


## 3. Standarisasi pada semua kolom numerik

Semua kolom fitur numerik distandarisasi dengan `StandardScaler` supaya memiliki rata-rata 0 dan simpangan baku 1. Kolom `diagnosis` adalah label, jadi tidak ikut distandarisasi.

In [7]:
# Ambil semua kolom numerik kecuali label
kolom_numerik = df.select_dtypes(include='number').columns.drop(label).tolist()
print(f'Jumlah kolom numerik yang distandarisasi: {len(kolom_numerik)}')

scaler = StandardScaler()
df[kolom_numerik] = scaler.fit_transform(df[kolom_numerik])

print('=== Hasil Setelah Standarisasi ===')
df.head()

Jumlah kolom numerik yang distandarisasi: 30
=== Hasil Setelah Standarisasi ===


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,1.097064,-2.073335,1.269934,0.984375,1.568466,3.283515,2.652874,2.532475,2.217515,...,1.886690,-1.359293,2.303601,2.001237,1.307686,2.616665,2.109526,2.296076,2.750622,1.937015
1,1,1.829821,-0.353632,1.685955,1.908708,-0.826962,-0.487072,-0.023846,0.548144,0.001392,...,1.805927,-0.369203,1.535126,1.890489,-0.375612,-0.430444,-0.146749,1.087084,-0.243890,0.281190
2,1,1.579888,0.456187,1.566503,1.558884,0.942210,1.052926,1.363478,2.037231,0.939685,...,1.511870,-0.023974,1.347475,1.456285,0.527407,1.082932,0.854974,1.955000,1.152255,0.201391
3,1,-0.768909,0.253732,-0.592687,-0.764464,3.283553,3.402909,1.915897,1.451707,2.867383,...,-0.281464,0.133984,-0.249939,-0.550021,3.394275,3.893397,1.989588,2.175786,6.046041,4.935010
4,1,1.750297,-1.151816,1.776573,1.826229,0.280372,0.539340,1.371011,1.428493,-0.009560,...,1.298575,-1.466770,1.338539,1.220724,0.220556,-0.313395,0.613179,0.729259,-0.868353,-0.397100


In [8]:
# Cek hasil standarisasi: rata-rata mendekati 0 dan simpangan baku mendekati 1
print('Rata-rata tiap kolom fitur:')
print(df[kolom_numerik].mean().round(4).head(), "\n")
print('Simpangan baku tiap kolom fitur:')
print(df[kolom_numerik].std().round(4).head())

Rata-rata tiap kolom fitur:
radius_mean       -0.0
texture_mean       0.0
perimeter_mean    -0.0
area_mean         -0.0
smoothness_mean   -0.0
dtype: float64 

Simpangan baku tiap kolom fitur:
radius_mean        1.0009
texture_mean       1.0009
perimeter_mean     1.0009
area_mean          1.0009
smoothness_mean    1.0009
dtype: float64


## 4. Stratified split data latih dan data uji (80 : 20)

Sama seperti praktikum, split dilakukan dengan `train_test_split` dan parameter `stratify` agar jumlah data Malignan dan Benign tetap proporsional.

In [9]:
X = df[fitur]
y = df[label]

# Stratified split dengan rasio 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print(f'Jumlah data asli: {df.shape[0]}')
print(f'Jumlah data train: {X_train.shape[0]}')
print(f'Jumlah data test: {X_test.shape[0]}')
print('=========')
print(f'Jumlah label data asli:\n{y.value_counts()}')
print(f'Jumlah label data train:\n{y_train.value_counts()}')
print(f'Jumlah label data test:\n{y_test.value_counts()}')

Jumlah data asli: 569
Jumlah data train: 455
Jumlah data test: 114
Jumlah label data asli:
diagnosis
0    357
1    212
Name: count, dtype: int64
Jumlah label data train:
diagnosis
0    285
1    170
Name: count, dtype: int64
Jumlah label data test:
diagnosis
0    72
1    42
Name: count, dtype: int64


In [10]:
# Simpan hasil preprocessing
df.to_csv('dataset/wbc-preprocessed.csv', index=False)
print('Hasil preprocessing tersimpan di dataset/wbc-preprocessed.csv')

Hasil preprocessing tersimpan di dataset/wbc-preprocessed.csv
